# Memory Experiment — PQRM Code

Punctured Quantum Reed-Muller codes with transversal T gate.  
Covers: RM generator matrix, patch construction, hypercube encoding, and Z-stabilizer memory experiment.

In [ ]:
import sys
from pathlib import Path
import numpy as np

ROOT = Path("../..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import stim
from lightstim.ir.builder import CircuitBuilder
from lightstim.ir.tracker import SyndromeTracker
from lightstim.ir.qec_system import QECSystem
from lightstim.qec_code.PQRM import PQRMPatch, PQRMLogicalOpSet, PQRMExtractionBlock
from lightstim.qec_code.PQRM.pqrm_patch import RM_generator_matrix, LOG_PQRM_LEN_DICT


## 1. RM Generator Matrix

In [ ]:
r, m = 1, 4
G, _ = RM_generator_matrix(r=r, m=m, variation="shortened")
print(f"RM({r},{m}) shortened: shape {G.shape}")
print(G)
print()
print("Supported (rx, rz, m):", sorted(LOG_PQRM_LEN_DICT.keys()))


## 2. PQRMPatch

In [ ]:
patch = PQRMPatch(rx=1, rz=2, m=4, punctured=True, shift=(0, 0))
info = patch.get_info()
for k, v in info.items():
    if k not in ("stabilizers", "logical_ops", "index_map", "qubit_coords", "coord_to_index"):
        print(f"  {k}: {v}")

z_stabs = [s for s in patch.stabilizers if s["type"] == "Z"]
x_stabs = [s for s in patch.stabilizers if s["type"] == "X"]
print(f"\nZ stabilizers (measured): {len(z_stabs)}")
print(f"X stabilizers (post-select): {len(x_stabs)}")
print(f"Logical operators: {len(patch.logical_ops)}")


## 3. Hypercube Encoding Circuit

In [ ]:
def build_encoding_circuit(patch, target_state="Z"):
    system = QECSystem()
    system.add_patch(patch, name="pqrm")
    tracker = SyndromeTracker(num_qubits=system.num_qubits, expected_num_logicals=system.num_logicals)
    builder = CircuitBuilder(tracker=tracker, system_config=system, if_detector=False)
    builder.write_coordinates()
    op_set = PQRMLogicalOpSet()
    op_set.hypercube_encode(builder, patch, target_state=target_state, patch_name="pqrm")
    return builder.circuit, system

encode_circuit, system = build_encoding_circuit(patch, "Z")
print(f"Encoding circuit: {encode_circuit.num_qubits} qubits")
encode_circuit


In [ ]:
# encode_circuit.diagram("detslice-with-ops-svg")


## 4. PQRM Memory Experiment

In [ ]:
def build_pqrm_memory_circuit(patch, basis="Z", rounds=2):
    system = QECSystem()
    system.add_patch(patch, name="pqrm")
    tracker = SyndromeTracker(num_qubits=system.num_qubits, expected_num_logicals=system.num_logicals)
    builder = CircuitBuilder(tracker=tracker, system_config=system, if_detector=True)
    builder.write_coordinates()
    op_set = PQRMLogicalOpSet()
    op_set.hypercube_encode(builder, patch, target_state=basis, patch_name="pqrm")
    builder.stabilizer_canonicalization()
    se_block = PQRMExtractionBlock(system)
    builder.apply_syndrome_extraction(circuit_chunk=se_block.circuit, rounds=rounds)
    data_indices = sorted(system.data_indices)
    builder.apply_data_readout(final_measurements={q: basis for q in data_indices})
    return builder.circuit

for basis in ["Z", "X", "Y"]:
    circ = build_pqrm_memory_circuit(patch, basis=basis, rounds=2)
    print(f"PQRM(1,2,4) {basis}-basis: {circ.num_qubits} qubits, {circ.num_detectors} detectors")


In [ ]:
circ_z = build_pqrm_memory_circuit(patch, basis="Z", rounds=2)
# circ_z.diagram("detslice-with-ops-svg", filter_coords=["L0"])
circ_z


## 5. Larger PQRM Codes

In [ ]:
for params in [(1, 2, 4), (1, 3, 5), (1, 4, 6)]:
    p = PQRMPatch(rx=params[0], rz=params[1], m=params[2], punctured=True, shift=(0, 0))
    circ = build_pqrm_memory_circuit(p, basis="Z", rounds=2)
    print(f"PQRM{params}: {circ.num_qubits} qubits, {circ.num_detectors} detectors")
